In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = Path("data/agent_skills_sample.db") 

assert DB_PATH.exists(), f"Arquivo não encontrado: {DB_PATH.resolve()}"

con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row

# def q(sql, params=()):
#     """Executa SQL e devolve um DataFrame."""
#     return pd.read_sql_query(sql, con, params=params)

# pd.set_option("display.max_columns", 80)
# pd.set_option("display.width", 160)
# plt.rcParams["figure.figsize"] = (8, 4.5)

print(f"Conectado: {DB_PATH.resolve()}")
print(f"Tamanho: {DB_PATH.stat().st_size / 1e6:.1f} MB")

# 1. Exploração das tabelas do DB

Four tables. Every string is UTF-8; timestamps are ISO-8601 UTC.

In [ ]:
tabelas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)["name"].tolist()
print("Tabelas:", tabelas)

linhas = {t: pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", con).at[0, "n"] for t in tabelas}
pd.Series(linhas, name="linhas").to_frame()

In [ ]:
# Esquema completo de cada tabela: nome, tipo declarado, not-null, PK
for t in tabelas:
    info = pd.read_sql_query(f"PRAGMA table_info({t})", con)[["name", "type", "notnull", "pk"]]
    print(f"\n=== {t} ({len(info)} colunas) ===")
    print(info.to_string(index=False))

### `artifact_siblings` — files bundled with a representative skill

In [ ]:
df_artifact_siblings = pd.read_sql_query("SELECT * FROM artifact_siblings", con)

df_artifact_siblings.info()
# df_artifact_siblings.head()

### `artifacts` — one row per file occurrence

In [ ]:
df_artifacts = pd.read_sql_query("SELECT * FROM artifacts", con)

df_artifacts.info()
# df_artifacts.head()

### `mining_runs` — collection provenance

One row per collection run: query, start and end timestamps, result count.

In [ ]:
df_mining_runs = pd.read_sql_query("SELECT * FROM mining_runs", con)

# df_mining_runs.info()     
df_mining_runs.head()

### `repos` — one row per repository

In [ ]:
df_repos = pd.read_sql_query("SELECT * FROM repos", con)

df_repos.info()
# df_repos.head()

In [ ]:
df_sqlite_sequence = pd.read_sql_query("SELECT * FROM sqlite_sequence", con)

# df_sqlite_sequence.info()
df_sqlite_sequence.head()

## Some queries

In [ ]:
# Most copied skill contents in the sample:
df_most_copied_skills = pd.read_sql_query("""SELECT MAX(name) AS name, COUNT(*) AS copies
FROM artifacts
GROUP BY file_sha
ORDER BY copies DESC LIMIT 10;""", con)

print(df_most_copied_skills)

In [ ]:
# Skill count by repository language:
df_language_repo = pd.read_sql_query("""SELECT r.language, COUNT(*) AS skills
FROM artifacts a JOIN repos r ON r.full_name = a.repo_full_name
GROUP BY r.language
ORDER BY skills DESC LIMIT 10;""", con)

print(df_language_repo)

